# Data Ingestion

This notebook explains Step 1 of the MOTEL workflow: how raw source data is converted into the `unmapped_entity` staging format. Like the source-specific reFuel notebook, this notebook focuses on the process and the data contract rather than the implementation details.

## Workflow

1. load the simplified ingestion schemas
2. inspect the `unmapped_entity` contract
3. review a real example of ingested output
4. understand where source-specific ingestion pipelines live
5. hand off staged data to `2_harmonise/2_data_harmonisation.ipynb`


## Run Controls

Use `PREVIEW_ROWS` to control how many example records are shown when previewing staged unmapped entities.


In [1]:
PREVIEW_ROWS = 2


In [2]:
from pathlib import Path
import yaml


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "schema_human").is_dir() and (candidate / "motel-db").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Start the notebook from inside motel-platform."
    )


PROJECT_ROOT = find_project_root()
SCHEMA_DIR = PROJECT_ROOT / "schema_human"
EXAMPLE_UNMAPPED_PATH = PROJECT_ROOT / "motel-db" / "unmapped_entity" / "unmapped_entities_refuel.yaml"
REFUEL_NOTEBOOK_PATH = PROJECT_ROOT / "1_ingest" / "examples" / "refuel" / "ingestion_pipeline.ipynb"
HARMONISATION_NOTEBOOK_PATH = PROJECT_ROOT / "2_harmonise" / "2_data_harmonisation.ipynb"

print(f"Project root: {PROJECT_ROOT}")
print(f"Schema directory: {SCHEMA_DIR}")
print(f"Example staged data: {EXAMPLE_UNMAPPED_PATH}")
print(f"Source-specific example pipeline: {REFUEL_NOTEBOOK_PATH.relative_to(PROJECT_ROOT)}")
print(f"Next workflow step: {HARMONISATION_NOTEBOOK_PATH.relative_to(PROJECT_ROOT)}")


Project root: E:\Barton\repositories\motel-platform
Schema directory: E:\Barton\repositories\motel-platform\schema_human
Example staged data: E:\Barton\repositories\motel-platform\motel-db\unmapped_entity\unmapped_entities_refuel.yaml
Source-specific example pipeline: 1_ingest\examples\refuel\ingestion_pipeline.ipynb
Next workflow step: 2_harmonise\2_data_harmonisation.ipynb


## Load The Ingestion Contract

The `schema_human/` folder contains the human-readable blueprint for MOTEL records. The most important file for Step 1 is `schema_human/unmapped_entity.yaml`, because every ingestion pipeline should produce records that follow this shape.


In [3]:
schema_files = sorted(SCHEMA_DIR.rglob("*.yaml"))
schemas = {}
for schema_path in schema_files:
    relative_name = schema_path.relative_to(SCHEMA_DIR).as_posix()
    with open(schema_path, "r", encoding="utf-8") as f:
        schemas[relative_name] = yaml.safe_load(f)

schema_unmapped = schemas["unmapped_entity.yaml"]

print("Available simplified schemas:")
for relative_name, schema in schemas.items():
    title = schema.get("title", "") if isinstance(schema, dict) else ""
    print(f"- {relative_name}" + (f"  ({title})" if title else ""))


Available simplified schemas:
- controlled_vocabulary/attribute.yaml
- controlled_vocabulary/capacity_scope.yaml
- controlled_vocabulary/carrier.yaml
- controlled_vocabulary/geographic_scope.yaml
- controlled_vocabulary/system_boundary.yaml
- controlled_vocabulary/temporal_scope.yaml
- linked_entity.yaml
- secondary/process.yaml
- secondary/source.yaml
- secondary/technology.yaml
- supplementary/contributor.yaml
- supplementary/review.yaml
- unmapped_entity.yaml


## Inspect The `unmapped_entity` Structure

This is the Step 1 data contract. At ingestion time, the goal is not to fully harmonise the data yet. Instead, the goal is to preserve the raw technology, scope, source, attribute, and balancing information in a structured staging format.


In [4]:
def print_tree(node, name="root", prefix=""):
    if prefix == "":
        print(name)

    if isinstance(node, dict):
        items = list(node.items())
        for i, (key, value) in enumerate(items):
            is_last = i == len(items) - 1
            branch = "└── " if is_last else "├── "
            next_prefix = prefix + ("    " if is_last else "│   ")

            if isinstance(value, dict):
                print(f"{prefix}{branch}{key}")
                print_tree(value, prefix=next_prefix)
            elif isinstance(value, list):
                print(f"{prefix}{branch}{key}: {', '.join(map(str, value))}")
            else:
                print(f"{prefix}{branch}{key}: {value}")

    elif isinstance(node, list):
        for i, value in enumerate(node):
            is_last = i == len(node) - 1
            branch = "└── " if is_last else "├── "
            print(f"{prefix}{branch}{value}")


print("Top-level unmapped_entity sections:")
for key in schema_unmapped.get("UnmappedEntity", {}):
    print(f"- {key}")

print()
print_tree(schema_unmapped, name="unmapped_entity.yaml")


Top-level unmapped_entity sections:
- technology_name
- technology
- scope
- sources
- attributes
- balancing
- metadata
- harmonisation_record

unmapped_entity.yaml
└── UnmappedEntity
    ├── technology_name: [Required; String] The literal unparsed common name or label of the technology as specified in the raw source material.
    ├── technology
    │   ├── technology_description: [String] Free-text engineering parameters describing the technical baseline of the asset.
    │   ├── technology_type: [String] The structural archetype classifying how the physical asset interacts with energy streams. Suggested: ['conversion', 'storage', 'distribution', 'consumption']
    │   ├── technology_category: [String] Broad domain or energy paradigm grouping for the hardware. Suggested: ['renewable', 'fossil_fuel', 'nuclear', 'synthetic']
    │   ├── technology_assumption: [String] Explicit technical constraints, design variations, or baseline modifications assumed by the raw source provider.
    │ 

## What A Good Ingestion Produces

A good ingestion pipeline writes YAML records into `motel-db/unmapped_entity/`. These records are still raw and source-shaped, but they are structured enough for the harmonisation notebook to process them systematically.


In [5]:
with open(EXAMPLE_UNMAPPED_PATH, "r", encoding="utf-8") as f:
    example_entities = yaml.safe_load(f)

print(f"Example staged entities loaded: {len(example_entities)}")
print()
print("Top-level keys in one staged record:")
for key in example_entities[0].keys():
    print(f"- {key}")

example_entities[:PREVIEW_ROWS]


Example staged entities loaded: 186

Top-level keys in one staged record:
- technology_name
- technology
- scope
- metadata
- sources
- attributes
- balancing
- mapping_status


[{'technology_name': 'NH3_CCGT_El',
  'technology': {'technology_description': None,
   'technology_type': 'Conversion',
   'technology_category': 'Gas turbine',
   'technology_assumption': None,
   'process_name': 'CCGT',
   'process_type': None,
   'process_category': None,
   'process_assumption': None},
  'scope': {'geographic_scope_description': 'ECA',
   'temporal_scope_description': '2050',
   'capacity_scope_description': None,
   'system_boundary_description': 'Plant ready to operate',
   'scope_assumption': 'Mature'},
  'metadata': {'related_project': 'reFuel.ch',
   'tags': ['Switzerland', 'power-to-X'],
   'other_notes': ['This unmapped entity was generated from the reFuel TechDatabase (2026-06-03) using the MOTEL ingestion pipeline.']},
  'sources': [{'source_name': 'report_power_to_ammonia_2018',
    'source_description': 'Power-to-ammonia in future North European 100 %  renewable power and heat system',
    'source_type': 'other',
    'link': 'https://doi.org/10.1016/j.i

## Where To Build Your Own Ingestion Pipeline

The implementation for a real ingestion pipeline should live in a source-specific folder under `1_ingest/ingestion_space/`. The reFuel example shows the recommended pattern:

1. keep the notebook short and workflow-oriented
2. move transformation logic into a helper module
3. load the source workbook or raw files
4. convert them into `unmapped_entity` records
5. export source-level YAML files and publish one combined staging file


In [6]:
print("Recommended source-specific ingestion workspace:")
print("- notebook: 1_ingest/examples/<source>/ingestion_pipeline.ipynb")
print("- helper:   1_ingest/examples/<source>/scripts/ingestion_helper.py")
print()
print("Reference implementation:")
print(f"- {REFUEL_NOTEBOOK_PATH.relative_to(PROJECT_ROOT)}")


Recommended source-specific ingestion workspace:
- notebook: 1_ingest/examples/<source>/ingestion_pipeline.ipynb
- helper:   1_ingest/examples/<source>/scripts/ingestion_helper.py

Reference implementation:
- 1_ingest\examples\refuel\ingestion_pipeline.ipynb


## Handoff To Harmonisation

Once your staged YAML exists in `motel-db/unmapped_entity/`, Step 1 is complete. The next step is `2_harmonise/2_data_harmonisation.ipynb`, which maps these raw staged records into linked entities, controlled vocabularies, and database tables.


In [7]:
print("Step 1 output:")
print(f"- {EXAMPLE_UNMAPPED_PATH.relative_to(PROJECT_ROOT)}")
print()
print("Step 2 notebook:")
print(f"- {HARMONISATION_NOTEBOOK_PATH.relative_to(PROJECT_ROOT)}")


Step 1 output:
- motel-db\unmapped_entity\unmapped_entities_refuel.yaml

Step 2 notebook:
- 2_harmonise\2_data_harmonisation.ipynb
